## Устанавливаем необходимые зависимости

In [13]:
%pip install -U langchain-core langchain-openai langchain-google-genai langchain-huggingface langchain-community gigachat python-dotenv


  Using cached langchain_openai-1.1.10-py3-none-any.whl.metadata (3.1 kB)
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_classic-1.0.1-py3-none-any.whl.metadata (4.2 kB)
  Using cached sqlalchemy-2.0.46-cp312-cp312-win_amd64.whl.metadata (9.8 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached numpy-2.4.2-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
  Using cached marshmallow-3.26.2-py3-none-any.whl.metadata (7.3 kB)
  Using cached typing_inspect-0.9.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached langchain_text_splitters-1.1.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached greenlet-3.3.1-cp312-cp312-win_amd64.whl.metadata (3.8 kB)
  Using cached mypy_extensions-1.1.0-py3-none-any.whl.metadata (1.1 kB)
Using cached langchain_openai-1.1.10-py3-none-any.whl (87 kB)
   ---------------------------------------- 0.0/66.5 kB ? eta


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Класс подключения моделей по API

In [ ]:
import os
import logging
from dotenv import load_dotenv

# Импорты LangChain
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_community.chat_models import GigaChat

# Настройка логирования
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s: %(message)s")
logger = logging.getLogger("UnifiedLLM")

load_dotenv(override=True)

class LLMPipeline:
    def __init__(self):

        self._models = {}

    def _get_model(self, provider: str, temperature: float = 0.7):
        """Фабричный метод для создания подключения к нужной модели."""

        cache_key = f"{provider}_{temperature}"
        if cache_key in self._models:
            return self._models[cache_key]

        model = None

        if provider == "gemini":
            if not os.getenv("GEMINI_API_KEY"):
                raise ValueError("GOOGLE_API_KEY не найден")
            
            model = ChatGoogleGenerativeAI(
                model="gemini-2.5-flash",
                temperature=temperature,
                convert_system_message_to_human=True 
            )

        #  2. DEEPSEEK 
        elif provider == "deepseek":
            if not os.getenv("DEEPSEEK_API_KEY"):
                raise ValueError("DEEPSEEK_API_KEY не найден")

            model = ChatOpenAI(
                model="deepseek-chat",
                api_key=os.getenv("DEEPSEEK_API_KEY"),
                base_url="https://api.deepseek.com",
                temperature=temperature
            )

        # 3. GIGACHAT 
        elif provider == "gigachat":
            if not os.getenv("GIGACHAT_CREDENTIALS"):
                raise ValueError("GIGACHAT_CREDENTIALS не найден")
            
            model = GigaChat(
                credentials=os.getenv("GIGACHAT_CREDENTIALS"),
                verify_ssl_certs=False, 
                model="GigaChat-Pro",
                temperature=temperature
            )

        # 4. HUGGING FACE 
        elif provider in ["llama", "qwen"]:
            hf_token = os.getenv("HUGGINGFACE_API_KEY")
            if not os.getenv("HUGGINGFACE_API_KEY"):
                raise ValueError("HUGGINGFACE_API_KEY не найден")

            repo_id = ""
            if provider == "llama":
                repo_id = "meta-llama/Meta-Llama-3-8B-Instruct"
            elif provider == "qwen":
                repo_id = "Qwen/Qwen2.5-7B-Instruct"

            
            llm = HuggingFaceEndpoint(
                repo_id=repo_id,
                task="text-generation",
                huggingfacehub_api_token=hf_token,
                temperature=0.1 if temperature < 0.1 else temperature, # HF не любит 0
                max_new_tokens=512
            )
            
            model = ChatHuggingFace(llm=llm)

        else:
            raise ValueError(f"Провайдер '{provider}' не поддерживается.")

        self._models[cache_key] = model
        return model

    def get_response(self, provider: str, prompt: str, system_prompt: str = "You are a helpful assistant."):
        """
        Универсальный метод получения ответа.
        """
        logger.info(f"--- Запрос к {provider} ---")
        
        try:
            model = self._get_model(provider)
            
            # Формируем список сообщений (LangChain стандарт)
            messages = [
                SystemMessage(content=system_prompt),
                HumanMessage(content=prompt)
            ]

            # Вызов модели
            response = model.invoke(messages)
            
            # response.content содержит текст ответа
            return response.content

        except Exception as e:
            logger.error(f"Ошибка при вызове {provider}: {e}")
            return f"Error: {str(e)}"

if __name__ == "__main__":
    bot = LLMPipeline()

    # Тест Gemini
    #print("Gemini:", bot.get_response("gemini", "Привет! Как дела?"))

    # Тест DeepSeek
    #print("DeepSeek:", bot.get_response("deepseek", "Напиши код на Python для сортировки пузырьком"))

    # Тест GigaChat (если есть ключ)
    #print("GigaChat:", bot.get_response("gigachat", "Расскажи про Сбер"))
    
    # Тест HuggingFace (Qwen)
    print(":", bot.get_response("qwen", "Hello! Who are you?"))

2026-02-20 16:54:42,289 INFO: --- Запрос к qwen ---


: Hello! I'm a large language model created by Alibaba Cloud. I go by the name Qwen. How can I assist you today?
